# Simulated Dual-Pol SICD H/Alpha Decomposition in GRDK Viewer



This notebook follows the same high-level pattern as `sentinel1_halpha_decomposition.py`, but uses simulated dual-pol channels (VV and VH) written as real SICD NITF files so it can run without an actual sensor collect.



Workflow:

1. Simulate two complex SAR channels and write them to disk as formal SICD NITF files.

2. Read the pair back using `SICDCollectionReader`, which returns a CYX cube with per-channel polarization metadata.

3. Run `DualPolHAlpha` decomposition.

4. Display co-pol magnitude (left pane) and H/Alpha RGB composite (right pane) side-by-side in GRDK Viewer.

In [ ]:
from datetime import datetime
from pathlib import Path
import logging
import sys
import numpy as np
from grdl.IO.sar import SICDCollectionReader, open_sicd_collection, SICDWriter
from grdl.IO.models.sicd import (
    SICDMetadata,
    SICDCollectionInfo,
    SICDImageData,
    SICDGeoData,
    SICDSCP,
    SICDGrid,
    SICDDirParam,
    SICDRadarCollection,
    SICDRcvChannel,
    SICDSCPCOA,
    SICDTimeline,
)
from grdl.IO.models.common import LatLon, LatLonHAE, RowCol, XYZ
from grdl.image_processing.decomposition.dual_pol_halpha import DualPolHAlpha

In [ ]:
# Section 1: Simulate a dual-pol SICD pair (VV + VH) and write as SICD NITF files
rng = np.random.default_rng(42)
rows, cols = 1024, 1024
rr, cc = np.indices((rows, cols), dtype=np.float32)
row_n = rr / rows
col_n = cc / cols
# Smooth scene structure + deterministic bright targets
base_amp = (
    0.2
    + 0.35 * np.sin(2.0 * np.pi * row_n * 1.7) ** 2
    + 0.25 * np.cos(2.0 * np.pi * col_n * 2.3) ** 2
)
target1 = np.exp(-(((row_n - 0.35) ** 2) + ((col_n - 0.6) ** 2)) / (2.0 * 0.01 ** 2))
target2 = np.exp(-(((row_n - 0.7) ** 2) + ((col_n - 0.25) ** 2)) / (2.0 * 0.015 ** 2))
amp_vv = base_amp + 2.5 * target1 + 1.8 * target2
amp_vh = 0.45 * base_amp + 0.9 * target1 + 0.6 * target2
# Speckle-like multiplicative variation and random phase
speckle_vv = rng.rayleigh(scale=0.7, size=(rows, cols)).astype(np.float32)
speckle_vh = rng.rayleigh(scale=0.85, size=(rows, cols)).astype(np.float32)
phase_vv = rng.uniform(-np.pi, np.pi, size=(rows, cols)).astype(np.float32)
phase_vh = (phase_vv + rng.normal(0.0, 0.35, size=(rows, cols))).astype(np.float32)
s_vv = (amp_vv * speckle_vv * np.exp(1j * phase_vv)).astype(np.complex64)
s_vh = (amp_vh * speckle_vh * np.exp(1j * phase_vh)).astype(np.complex64)
out_dir = Path.cwd() / "output_simulated_sicd_dualpol"
out_dir.mkdir(parents=True, exist_ok=True)
vv_path = out_dir / "simulated_vv.nitf"
vh_path = out_dir / "simulated_vh.nitf"
collect_start = datetime(2025, 1, 1, 12, 0, 0)
def _make_sicd_meta(n_rows, n_cols, tx_rcv_pol, core_name):
    """Build minimal SICDMetadata sufficient for SICDWriter and SICDCollectionReader."""
    scp_ecf = XYZ(x=6378137.0, y=0.0, z=0.0)
    scp_llh = LatLonHAE(lat=0.0, lon=0.0, hae=0.0)
    corners = [
        LatLon(lat=0.01, lon=-0.01),
        LatLon(lat=0.01, lon=0.01),
        LatLon(lat=-0.01, lon=0.01),
        LatLon(lat=-0.01, lon=-0.01),
    ]
    return SICDMetadata(
        format="SICD",
        rows=n_rows,
        cols=n_cols,
        dtype="complex64",
        image_data=SICDImageData(
            num_rows=n_rows,
            num_cols=n_cols,
            pixel_type="RE32F_IM32F",
            scp_pixel=RowCol(row=n_rows // 2, col=n_cols // 2),
        ),
        geo_data=SICDGeoData(
            scp=SICDSCP(ecf=scp_ecf, llh=scp_llh),
            image_corners=corners,
        ),
        grid=SICDGrid(
            image_plane="GROUND",
            type="RGAZIM",
            row=SICDDirParam(uvect_ecf=XYZ(x=1.0, y=0.0, z=0.0), ss=1.0),
            col=SICDDirParam(uvect_ecf=XYZ(x=0.0, y=1.0, z=0.0), ss=1.0),
        ),
        collection_info=SICDCollectionInfo(
            collector_name="SIM",
            core_name=core_name,
            classification="U",
        ),
        timeline=SICDTimeline(
            collect_start=collect_start,
            collect_duration=1.0,
        ),
        scpcoa=SICDSCPCOA(
            scp_time=0.0,
            arp_pos=XYZ(x=6378137.0, y=-15000.0, z=700000.0),
            arp_vel=XYZ(x=0.0, y=7500.0, z=0.0),
            side_of_track="R",
        ),
        radar_collection=SICDRadarCollection(
            tx_polarization=tx_rcv_pol.split(":")[0],
            rcv_channels=[SICDRcvChannel(tx_rcv_polarization=tx_rcv_pol)],
        ),
    )
sarpy_naming_logger = logging.getLogger("sarpy.io.complex.naming.utils")
previous_sarpy_naming_level = sarpy_naming_logger.level
sarpy_naming_logger.setLevel(logging.CRITICAL)
try:
    for path, data, tx_rcv_pol, core in [
        (vv_path, s_vv, "V:V", "SIM_VV"),
        (vh_path, s_vh, "V:H", "SIM_VH"),
    ]:
        meta = _make_sicd_meta(rows, cols, tx_rcv_pol, core)
        writer = SICDWriter(path, metadata=meta)
        writer.write(data)
        print(f"Wrote {path.name}  ({path.stat().st_size / 1e6:.1f} MB)")
finally:
    sarpy_naming_logger.setLevel(previous_sarpy_naming_level)
print(f"\nSICD files written to: {out_dir}")
print(f"  VV: {vv_path.name}")
print(f"  VH: {vh_path.name}")

In [ ]:
# Section 2: Read the SICD pair via SICDCollectionReader -> CYX cube
# SICDCollectionReader orders channels in the order provided.
reader = open_sicd_collection([vv_path, vh_path])
pols = [cm.polarization for cm in reader.metadata.channel_metadata]
print(f"Collection reader opened:")
print(f"  files        : {[vv_path.name, vh_path.name]}")
print(f"  polarizations: {pols}")
print(f"  shape        : {reader.get_shape()}")
print(f"  axis_order   : {reader.metadata.axis_order}")
cube = reader.read_full()   # (2, rows, cols) complex64 CYX
metadata = reader.metadata
print(f"\ncube.shape={cube.shape}  dtype={cube.dtype}")
print(f"co-pol  (VV) power range: [{np.abs(cube[0]).min():.4f}, {np.abs(cube[0]).max():.4f}]")
print(f"cross-pol (VH) power range: [{np.abs(cube[1]).min():.4f}, {np.abs(cube[1]).max():.4f}]")

In [ ]:
# Section 3: Run dual-pol H/Alpha decomposition
# DualPolHAlpha.execute() reads axis_order and channel_metadata from the
# SICDCollectionReader metadata to find co-pol (index 0) and cross-pol (index 1).
halpha = DualPolHAlpha(window_size=7)
components, updated_meta = halpha.execute(metadata, cube)
rgb_float, rgb_meta = halpha.to_rgb(components)   # (3, rows, cols) float32 [0, 1]
print("Decomposition complete")
print(f"  entropy range    : [{components['entropy'].min():.3f}, {components['entropy'].max():.3f}]")
print(f"  alpha range (deg): [{components['alpha'].min():.2f}, {components['alpha'].max():.2f}]")
print(f"  anisotropy range : [{components['anisotropy'].min():.3f}, {components['anisotropy'].max():.3f}]")
print(f"  span range       : [{components['span'].min():.3e}, {components['span'].max():.3e}]")
print(f"  rgb_float shape  : {rgb_float.shape}")

In [ ]:
# Section 4: Display in GRDK Viewer - co-pol magnitude (left) vs H/Alpha RGB (right)
from PyQt6.QtWidgets import QApplication
from grdk.viewers import ViewerMainWindow
app = QApplication.instance() or QApplication(sys.argv)
# Co-pol magnitude for the left pane (single-band float32)
copol_mag = np.abs(cube[0]).astype(np.float32)   # (rows, cols)
window = ViewerMainWindow()
window.setWindowTitle("Simulated SICD Dual-Pol H/Alpha")
# Left pane: co-pol |S_VV| magnitude
window.set_array(copol_mag, title="Co-pol |S_VV|", pane=0)
# Right pane: H/Alpha RGB composite - (3, rows, cols) CYX float32 [0, 1]
window.set_array(rgb_float, title="H/Alpha RGB", pane=1)
# Activate dual-pane mode
window._viewer.set_mode("dual")
window.show()
app.exec()